# HPRC Ensembl vs CAT Annotation QC Report

This notebook generates a comprehensive QC report comparing Ensembl (linear projection) and CAT (graph-based projection) gene annotations across 400+ HPRC assemblies.

**Memory-efficient design**: Processes data in chunks, computes summaries, and saves to disk immediately.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import gc
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Configuration

In [ ]:
# Set paths
OUTPUT_DIR = Path('../results')  # Adjust this path as needed
QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUMMARY_DIR = OUTPUT_DIR / 'summary_stats'  # Where we'll save intermediate results
SUMMARY_DIR.mkdir(exist_ok=True, parents=True)

# Processing parameters
CHUNK_SIZE = 50  # Process 50 assemblies at a time

print(f"Output directory: {OUTPUT_DIR}")
print(f"QC metrics directory: {QC_DIR}")
print(f"Summary stats directory: {SUMMARY_DIR}")
print(f"Chunk size: {CHUNK_SIZE} assemblies")

## 1. Process Transcript Concordance (Chunked)

Process files in chunks to avoid memory issues.

In [ ]:
def process_transcript_concordance_chunked():
    """Process transcript concordance in chunks and save summary stats."""
    files = list(QC_DIR.rglob('*_transcript_concordance.tsv'))
    print(f"Found {len(files)} transcript concordance files")
    
    if not files:
        return None
    
    # Process in chunks
    per_assembly_stats = []
    concordance_rates = []
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1} ({len(chunk_files)} files)")
        
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                chunk_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
        
        if chunk_dfs:
            chunk_data = pd.concat(chunk_dfs, ignore_index=True)
            
            # Extract concordance rates for overall distribution
            concordance_rates.extend(chunk_data['transcript_concordance_rate'].tolist())
            
            # Per-assembly summary
            assembly_summary = chunk_data.groupby('assembly_accession').agg({
                'transcript_concordance_rate': 'mean',
                'n_exact_matches': 'sum',
                'n_partial_matches': 'sum',
                'ensembl_gene_id': 'count'
            }).reset_index()
            per_assembly_stats.append(assembly_summary)
            
            # Clear memory
            del chunk_data, chunk_dfs
            gc.collect()
    
    # Combine and save per-assembly stats
    if per_assembly_stats:
        per_assembly_df = pd.concat(per_assembly_stats, ignore_index=True)
        per_assembly_df.columns = ['assembly_accession', 'mean_concordance_rate', 
                                    'total_exact_matches', 'total_partial_matches', 'n_genes']
        per_assembly_df.to_csv(SUMMARY_DIR / 'transcript_concordance_per_assembly.tsv', 
                               sep='\t', index=False)
        print(f"Saved per-assembly stats: {len(per_assembly_df)} assemblies")
    
    # Save concordance rate distribution
    concordance_df = pd.DataFrame({'concordance_rate': concordance_rates})
    concordance_df.to_csv(SUMMARY_DIR / 'transcript_concordance_rates.tsv', 
                          sep='\t', index=False)
    print(f"Saved concordance rates: {len(concordance_rates)} gene pairs")
    
    # Clear memory
    del concordance_rates, per_assembly_stats
    gc.collect()
    
    return per_assembly_df

transcript_summary = process_transcript_concordance_chunked()
print("\n✓ Transcript concordance processing complete")

## 2. Process Coding Integrity (Chunked)

In [ ]:
def process_coding_integrity_chunked():
    """Process coding integrity in chunks and save summary stats."""
    files = list(QC_DIR.rglob('*_coding_integrity.tsv'))
    print(f"Found {len(files)} coding integrity files")
    
    if not files:
        return None
    
    # Accumulators
    total_genes = 0
    genes_with_cds = 0
    start_matches = 0
    stop_matches = 0
    both_matches = 0
    frameshifts = 0
    length_diffs = []
    per_assembly_stats = []
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1}")
        
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                # Convert boolean strings
                for col in ['start_codon_match', 'stop_codon_match', 'frameshift_detected', 
                           'has_ensembl_cds', 'has_cat_cds']:
                    if col in df.columns:
                        df[col] = df[col].map({'True': True, 'False': False, True: True, False: False})
                chunk_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
        
        if chunk_dfs:
            chunk_data = pd.concat(chunk_dfs, ignore_index=True)
            
            # Filter to genes with CDS in both
            with_cds = chunk_data[
                (chunk_data['has_ensembl_cds'] == True) & 
                (chunk_data['has_cat_cds'] == True)
            ]
            
            # Accumulate stats
            total_genes += len(chunk_data)
            genes_with_cds += len(with_cds)
            
            if len(with_cds) > 0:
                start_matches += (with_cds['start_codon_match'] == True).sum()
                stop_matches += (with_cds['stop_codon_match'] == True).sum()
                both_matches += ((with_cds['start_codon_match'] == True) & 
                                (with_cds['stop_codon_match'] == True)).sum()
                frameshifts += (with_cds['frameshift_detected'] == True).sum()
                
                # Sample length differences (don't keep all)
                length_diffs.extend(with_cds['length_difference'].sample(
                    min(1000, len(with_cds))
                ).tolist())
                
                # Per-assembly stats
                assembly_stats = with_cds.groupby('assembly_accession').apply(
                    lambda x: pd.Series({
                        'n_protein_coding': len(x),
                        'start_stop_agreement': ((x['start_codon_match'] == True) & 
                                                (x['stop_codon_match'] == True)).sum() / len(x),
                        'n_frameshifts': (x['frameshift_detected'] == True).sum()
                    })
                ).reset_index()
                per_assembly_stats.append(assembly_stats)
            
            del chunk_data, chunk_dfs, with_cds
            gc.collect()
    
    # Save overall stats
    overall_stats = pd.DataFrame([{
        'total_genes': total_genes,
        'genes_with_cds_both': genes_with_cds,
        'start_codon_matches': start_matches,
        'stop_codon_matches': stop_matches,
        'both_codons_match': both_matches,
        'frameshifts_detected': frameshifts,
        'pct_with_cds': genes_with_cds / total_genes if total_genes > 0 else 0,
        'pct_start_match': start_matches / genes_with_cds if genes_with_cds > 0 else 0,
        'pct_stop_match': stop_matches / genes_with_cds if genes_with_cds > 0 else 0,
        'pct_both_match': both_matches / genes_with_cds if genes_with_cds > 0 else 0,
        'pct_frameshift': frameshifts / genes_with_cds if genes_with_cds > 0 else 0
    }])
    overall_stats.to_csv(SUMMARY_DIR / 'coding_integrity_overall.tsv', sep='\t', index=False)
    
    # Save length differences sample
    pd.DataFrame({'length_difference': length_diffs}).to_csv(
        SUMMARY_DIR / 'cds_length_differences_sample.tsv', sep='\t', index=False
    )
    
    # Save per-assembly stats
    if per_assembly_stats:
        per_assembly_df = pd.concat(per_assembly_stats, ignore_index=True)
        per_assembly_df.to_csv(SUMMARY_DIR / 'coding_integrity_per_assembly.tsv', 
                               sep='\t', index=False)
    
    print(f"Saved coding integrity stats: {total_genes} total genes, {genes_with_cds} with CDS")
    
    del length_diffs, per_assembly_stats
    gc.collect()
    
    return overall_stats

coding_summary = process_coding_integrity_chunked()
print("\n✓ Coding integrity processing complete")

## 3. Process Gene Presence/Absence (Chunked)

In [ ]:
def process_gene_presence_chunked():
    """Process gene presence in chunks and save summary stats."""
    files = list(QC_DIR.rglob('*_gene_presence.tsv'))
    print(f"Found {len(files)} gene presence files")
    
    if not files:
        return None
    
    # Count genes across all assemblies
    gene_counts = {'both': 0, 'ensembl_only': 0, 'cat_only': 0}
    ensembl_missing_genes = {}
    cat_missing_genes = {}
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1}")
        
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                # Convert boolean strings
                df['present_in_ensembl'] = df['present_in_ensembl'].map(
                    {'True': True, 'False': False, True: True, False: False}
                )
                df['present_in_cat'] = df['present_in_cat'].map(
                    {'True': True, 'False': False, True: True, False: False}
                )
                chunk_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
        
        if chunk_dfs:
            chunk_data = pd.concat(chunk_dfs, ignore_index=True)
            
            # Count by category
            both = ((chunk_data['present_in_ensembl'] == True) & 
                   (chunk_data['present_in_cat'] == True)).sum()
            ens_only = ((chunk_data['present_in_ensembl'] == True) & 
                       (chunk_data['present_in_cat'] == False)).sum()
            cat_only = ((chunk_data['present_in_ensembl'] == False) & 
                       (chunk_data['present_in_cat'] == True)).sum()
            
            gene_counts['both'] += both
            gene_counts['ensembl_only'] += ens_only
            gene_counts['cat_only'] += cat_only
            
            # Track top missing genes
            ens_missing = chunk_data[
                (chunk_data['present_in_ensembl'] == True) & 
                (chunk_data['present_in_cat'] == False)
            ]
            for gene in ens_missing['gene_name'].value_counts().head(50).index:
                ensembl_missing_genes[gene] = ensembl_missing_genes.get(gene, 0) + \
                    (ens_missing['gene_name'] == gene).sum()
            
            cat_missing = chunk_data[
                (chunk_data['present_in_ensembl'] == False) & 
                (chunk_data['present_in_cat'] == True)
            ]
            for gene in cat_missing['gene_name'].value_counts().head(50).index:
                cat_missing_genes[gene] = cat_missing_genes.get(gene, 0) + \
                    (cat_missing['gene_name'] == gene).sum()
            
            del chunk_data, chunk_dfs
            gc.collect()
    
    # Save summary
    summary = pd.DataFrame([gene_counts])
    total = sum(gene_counts.values())
    summary['total'] = total
    summary['pct_both'] = gene_counts['both'] / total if total > 0 else 0
    summary['pct_ensembl_only'] = gene_counts['ensembl_only'] / total if total > 0 else 0
    summary['pct_cat_only'] = gene_counts['cat_only'] / total if total > 0 else 0
    summary.to_csv(SUMMARY_DIR / 'gene_presence_summary.tsv', sep='\t', index=False)
    
    # Save top missing genes
    pd.DataFrame([
        {'gene': k, 'n_assemblies': v, 'missing_from': 'CAT'} 
        for k, v in sorted(ensembl_missing_genes.items(), key=lambda x: -x[1])[:50]
    ] + [
        {'gene': k, 'n_assemblies': v, 'missing_from': 'Ensembl'} 
        for k, v in sorted(cat_missing_genes.items(), key=lambda x: -x[1])[:50]
    ]).to_csv(SUMMARY_DIR / 'top_missing_genes.tsv', sep='\t', index=False)
    
    print(f"Saved gene presence stats: {total} total occurrences")
    
    del ensembl_missing_genes, cat_missing_genes
    gc.collect()
    
    return summary

presence_summary = process_gene_presence_chunked()
print("\n✓ Gene presence processing complete")

## 4. Process Multi-Mapping (Lightweight - just count)

In [ ]:
def process_multi_mapping_chunked():
    """Process multi-mapping and save counts."""
    files = list(QC_DIR.rglob('*_multi_mapping.tsv'))
    print(f"Found {len(files)} multi-mapping files")
    
    if not files:
        return None
    
    ensembl_counts = []
    cat_counts = []
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1}")
        
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                ensembl_multi = df[df['source'] == 'ensembl']
                cat_multi = df[df['source'] == 'cat']
                
                if len(ensembl_multi) > 0:
                    ensembl_counts.extend(ensembl_multi['n_matches'].tolist())
                if len(cat_multi) > 0:
                    cat_counts.extend(cat_multi['n_matches'].tolist())
                    
            except Exception as e:
                print(f"Error loading {f}: {e}")
    
    # Save summary
    summary = pd.DataFrame([{
        'ensembl_genes_multi_mapped': len(ensembl_counts),
        'cat_genes_multi_mapped': len(cat_counts),
        'mean_ensembl_matches': np.mean(ensembl_counts) if ensembl_counts else 0,
        'mean_cat_matches': np.mean(cat_counts) if cat_counts else 0,
        'max_ensembl_matches': max(ensembl_counts) if ensembl_counts else 0,
        'max_cat_matches': max(cat_counts) if cat_counts else 0
    }])
    summary.to_csv(SUMMARY_DIR / 'multi_mapping_summary.tsv', sep='\t', index=False)
    
    # Save distributions
    pd.DataFrame({'n_matches': ensembl_counts, 'source': 'ensembl'}).to_csv(
        SUMMARY_DIR / 'multi_mapping_ensembl_dist.tsv', sep='\t', index=False
    )
    pd.DataFrame({'n_matches': cat_counts, 'source': 'cat'}).to_csv(
        SUMMARY_DIR / 'multi_mapping_cat_dist.tsv', sep='\t', index=False
    )
    
    print(f"Saved multi-mapping stats: {len(ensembl_counts)} Ensembl, {len(cat_counts)} CAT")
    
    del ensembl_counts, cat_counts
    gc.collect()
    
    return summary

multi_summary = process_multi_mapping_chunked()
print("\n✓ Multi-mapping processing complete")

## 5. Process RBH Pairs (Sample for visualization)

In [ ]:
def process_rbh_pairs_chunked():
    """Process RBH pairs and save statistics + sample for plotting."""
    files = list(RESULTS_DIR.rglob('*.gene_pairs_rbh.tsv'))
    print(f"Found {len(files)} RBH files")
    
    if not files:
        return None
    
    total_pairs = 0
    high_overlap_count = 0
    classification_counts = {}
    coverage_sample = []
    SAMPLE_SIZE = 10000  # Sample for scatter plot
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1}")
        
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                chunk_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
        
        if chunk_dfs:
            chunk_data = pd.concat(chunk_dfs, ignore_index=True)
            
            total_pairs += len(chunk_data)
            high_overlap_count += ((chunk_data['frac_ensembl_covered'] >= 0.9) & 
                                   (chunk_data['frac_cat_covered'] >= 0.9)).sum()
            
            # Classification counts
            if 'classification' in chunk_data.columns:
                for cls, count in chunk_data['classification'].value_counts().items():
                    classification_counts[cls] = classification_counts.get(cls, 0) + count
            
            # Sample for scatter plot
            if len(coverage_sample) < SAMPLE_SIZE:
                sample = chunk_data[['frac_ensembl_covered', 'frac_cat_covered']].sample(
                    min(SAMPLE_SIZE - len(coverage_sample), len(chunk_data))
                )
                coverage_sample.append(sample)
            
            del chunk_data, chunk_dfs
            gc.collect()
    
    # Save summary
    summary = pd.DataFrame([{
        'total_rbh_pairs': total_pairs,
        'high_overlap_pairs': high_overlap_count,
        'pct_high_overlap': high_overlap_count / total_pairs if total_pairs > 0 else 0
    }])
    summary.to_csv(SUMMARY_DIR / 'rbh_pairs_summary.tsv', sep='\t', index=False)
    
    # Save classification breakdown
    pd.DataFrame([
        {'classification': k, 'count': v, 'percentage': v/total_pairs if total_pairs > 0 else 0}
        for k, v in classification_counts.items()
    ]).to_csv(SUMMARY_DIR / 'rbh_classifications.tsv', sep='\t', index=False)
    
    # Save coverage sample
    if coverage_sample:
        sample_df = pd.concat(coverage_sample, ignore_index=True)
        sample_df.to_csv(SUMMARY_DIR / 'rbh_coverage_sample.tsv', sep='\t', index=False)
    
    print(f"Saved RBH stats: {total_pairs} total pairs, {len(coverage_sample)} in sample")
    
    del coverage_sample, classification_counts
    gc.collect()
    
    return summary

rbh_summary = process_rbh_pairs_chunked()
print("\n✓ RBH pairs processing complete")

## 6. Generate Visualizations from Saved Data

Now load only the small summary files and generate plots.

In [ ]:
print("\n" + "="*80)
print("GENERATING SUMMARY REPORT")
print("="*80 + "\n")

# Load summary data (small files)
concordance_rates = pd.read_csv(SUMMARY_DIR / 'transcript_concordance_rates.tsv', sep='\t')
coding_overall = pd.read_csv(SUMMARY_DIR / 'coding_integrity_overall.tsv', sep='\t')
presence_summary = pd.read_csv(SUMMARY_DIR / 'gene_presence_summary.tsv', sep='\t')
multi_summary = pd.read_csv(SUMMARY_DIR / 'multi_mapping_summary.tsv', sep='\t')
rbh_summary = pd.read_csv(SUMMARY_DIR / 'rbh_pairs_summary.tsv', sep='\t')

print("Summary Statistics:")
print(f"  Total RBH pairs analyzed: {rbh_summary['total_rbh_pairs'].iloc[0]:,}")
print(f"  Gene pairs with transcript data: {len(concordance_rates):,}")
print(f"  Mean transcript concordance: {concordance_rates['concordance_rate'].mean():.2%}")
print(f"  Protein-coding genes with CDS: {coding_overall['genes_with_cds_both'].iloc[0]:,}")
print(f"  Start & stop codon agreement: {coding_overall['pct_both_match'].iloc[0]:.1%}")
print(f"  Multi-mapped Ensembl genes: {multi_summary['ensembl_genes_multi_mapped'].iloc[0]:,}")
print(f"  Multi-mapped CAT genes: {multi_summary['cat_genes_multi_mapped'].iloc[0]:,}")

### 6.1 Transcript Concordance Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of concordance rates
axes[0].hist(concordance_rates['concordance_rate'], bins=50, edgecolor='black')
mean_conc = concordance_rates['concordance_rate'].mean()
axes[0].axvline(mean_conc, color='red', linestyle='--', label=f'Mean: {mean_conc:.2%}')
axes[0].set_xlabel('Transcript Concordance Rate')
axes[0].set_ylabel('Number of Genes')
axes[0].set_title('Distribution of Transcript Concordance\n(Per RBH Gene Pair)')
axes[0].legend()

# Plot 2: Per-assembly distribution
per_assembly = pd.read_csv(SUMMARY_DIR / 'transcript_concordance_per_assembly.tsv', sep='\t')
axes[1].hist(per_assembly['mean_concordance_rate'], bins=30, edgecolor='black')
axes[1].axvline(per_assembly['mean_concordance_rate'].mean(), color='red', 
                linestyle='--', label=f'Mean: {per_assembly["mean_concordance_rate"].mean():.2%}')
axes[1].set_xlabel('Mean Concordance Rate')
axes[1].set_ylabel('Number of Assemblies')
axes[1].set_title(f'Per-Assembly Transcript Concordance\n({len(per_assembly)} assemblies)')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'transcript_concordance_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved transcript_concordance_summary.png")

### 6.2 Coding Integrity Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Codon agreement
stats = coding_overall.iloc[0]
categories = ['Start\nMatch', 'Stop\nMatch', 'Both\nMatch', 'Frameshift']
percentages = [
    stats['pct_start_match'] * 100,
    stats['pct_stop_match'] * 100,
    stats['pct_both_match'] * 100,
    stats['pct_frameshift'] * 100
]
colors = ['green', 'green', 'darkgreen', 'red']

axes[0].bar(categories, percentages, color=colors, edgecolor='black', alpha=0.7)
axes[0].set_ylabel('Percentage of Genes')
axes[0].set_title(f'Codon Position Agreement\n({int(stats["genes_with_cds_both"]):,} protein-coding genes)')
axes[0].set_ylim([0, 105])
for i, p in enumerate(percentages):
    axes[0].text(i, p+2, f'{p:.1f}%', ha='center', va='bottom')

# Plot 2: CDS length differences
length_diffs = pd.read_csv(SUMMARY_DIR / 'cds_length_differences_sample.tsv', sep='\t')
axes[1].hist(length_diffs['length_difference'], bins=50, edgecolor='black')
axes[1].set_xlabel('Absolute CDS Length Difference (bp)')
axes[1].set_ylabel('Number of Genes')
axes[1].set_title(f'CDS Length Differences\n(Sample of {len(length_diffs):,} genes)')
axes[1].axvline(0, color='red', linestyle='--', label='Perfect match')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coding_integrity_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved coding_integrity_summary.png")

### 6.3 Gene Presence/Absence & Multi-Mapping

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Gene presence/absence
presence = presence_summary.iloc[0]
categories = ['Both\nAnnotations', 'Ensembl\nOnly', 'CAT\nOnly']
counts = [presence['both'], presence['ensembl_only'], presence['cat_only']]
colors = ['green', 'blue', 'orange']

axes[0].bar(categories, counts, color=colors, edgecolor='black', alpha=0.7)
axes[0].set_ylabel('Number of Gene Occurrences')
axes[0].set_title('Gene Presence/Absence by Name\n(Across all assemblies)')
for i, c in enumerate(counts):
    pct = [presence['pct_both'], presence['pct_ensembl_only'], presence['pct_cat_only']][i] * 100
    axes[0].text(i, c, f'{int(c):,}\n({pct:.1f}%)', ha='center', va='bottom')

# Plot 2: Multi-mapping counts
multi = multi_summary.iloc[0]
categories = ['Ensembl\n1-to-many', 'CAT\nmany-to-1']
counts = [multi['ensembl_genes_multi_mapped'], multi['cat_genes_multi_mapped']]
colors = ['blue', 'orange']

axes[1].bar(categories, counts, color=colors, edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Number of Genes')
axes[1].set_title('Multi-Mapping Gene Counts')
for i, c in enumerate(counts):
    mean_matches = [multi['mean_ensembl_matches'], multi['mean_cat_matches']][i]
    axes[1].text(i, c, f'{int(c):,}\n(avg {mean_matches:.1f} matches)', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'presence_multimapping_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved presence_multimapping_summary.png")

### 6.4 RBH Overlap Quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Reciprocal coverage scatter (from sample)
coverage_sample = pd.read_csv(SUMMARY_DIR / 'rbh_coverage_sample.tsv', sep='\t')
axes[0].scatter(coverage_sample['frac_ensembl_covered'], 
               coverage_sample['frac_cat_covered'], alpha=0.3, s=10)
axes[0].axhline(0.9, color='red', linestyle='--', alpha=0.5, label='90% threshold')
axes[0].axvline(0.9, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Fraction of Ensembl Gene Covered')
axes[0].set_ylabel('Fraction of CAT Gene Covered')
axes[0].set_title(f'Reciprocal Coverage in RBH Pairs\n(Sample of {len(coverage_sample):,} pairs)')
axes[0].legend()
axes[0].set_xlim([0, 1.05])
axes[0].set_ylim([0, 1.05])

# Plot 2: Classification breakdown
classifications = pd.read_csv(SUMMARY_DIR / 'rbh_classifications.tsv', sep='\t')
classifications = classifications.sort_values('count', ascending=True)
axes[1].barh(range(len(classifications)), classifications['count'], edgecolor='black')
axes[1].set_yticks(range(len(classifications)))
axes[1].set_yticklabels(classifications['classification'])
axes[1].set_xlabel('Number of Gene Pairs')
axes[1].set_title('RBH Pair Overlap Classification')
for i, row in classifications.iterrows():
    idx = list(classifications.index).index(i)
    axes[1].text(row['count'], idx, f" {int(row['count']):,} ({row['percentage']*100:.1f}%)", va='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rbh_overlap_quality.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved rbh_overlap_quality.png")

## 7. Generate Final Summary Table

In [ ]:
# Create comprehensive summary table
summary_table = pd.DataFrame([
    ['Total RBH Pairs', f"{rbh_summary['total_rbh_pairs'].iloc[0]:,}", 
     f"{rbh_summary['pct_high_overlap'].iloc[0]:.1%}", '≥90% reciprocal overlap'],
    ['Transcript Concordance', f"{concordance_rates['concordance_rate'].mean():.1%}",
     f"{(concordance_rates['concordance_rate'] == 1.0).sum():,} / {len(concordance_rates):,}",
     'Mean rate; perfect matches'],
    ['Start & Stop Codons Match', f"{coding_overall['pct_both_match'].iloc[0]:.1%}",
     f"{int(coding_overall['both_codons_match'].iloc[0]):,} / {int(coding_overall['genes_with_cds_both'].iloc[0]):,}",
     'Protein-coding genes with CDS'],
    ['Potential Frameshifts', f"{coding_overall['pct_frameshift'].iloc[0]:.1%}",
     f"{int(coding_overall['frameshifts_detected'].iloc[0]):,}",
     'Non-divisible-by-3 length diffs'],
    ['Genes in Both Annotations', f"{presence_summary['pct_both'].iloc[0]:.1%}",
     f"{int(presence_summary['both'].iloc[0]):,} / {int(presence_summary['total'].iloc[0]):,}",
     'Named genes across all assemblies'],
    ['Multi-mapped Genes', f"{int(multi_summary['ensembl_genes_multi_mapped'].iloc[0] + multi_summary['cat_genes_multi_mapped'].iloc[0]):,}",
     f"E: {int(multi_summary['ensembl_genes_multi_mapped'].iloc[0]):,}, C: {int(multi_summary['cat_genes_multi_mapped'].iloc[0]):,}",
     '1-to-many or many-to-1']
], columns=['Metric', 'Value', 'Count', 'Description'])

print("\n" + "="*100)
print("FINAL QC SUMMARY TABLE")
print("="*100)
print(summary_table.to_string(index=False))
print("="*100)

summary_table.to_csv(OUTPUT_DIR / 'qc_summary_table.tsv', sep='\t', index=False)
print(f"\n✓ Summary table saved to: {OUTPUT_DIR / 'qc_summary_table.tsv'}")

## 8. Per-Assembly Summary for Downstream Analysis

In [ ]:
# Merge per-assembly summaries
transcript_per_asm = pd.read_csv(SUMMARY_DIR / 'transcript_concordance_per_assembly.tsv', sep='\t')
coding_per_asm = pd.read_csv(SUMMARY_DIR / 'coding_integrity_per_assembly.tsv', sep='\t')

per_assembly_final = transcript_per_asm.merge(
    coding_per_asm, on='assembly_accession', how='outer'
)

per_assembly_final.to_csv(OUTPUT_DIR / 'per_assembly_qc_summary.tsv', sep='\t', index=False)
print(f"✓ Per-assembly summary saved: {len(per_assembly_final)} assemblies")
print(f"  File: {OUTPUT_DIR / 'per_assembly_qc_summary.tsv'}")

## Summary

### Generated Files:
1. **Plots** (PNG):
   - `transcript_concordance_summary.png`
   - `coding_integrity_summary.png`
   - `presence_multimapping_summary.png`
   - `rbh_overlap_quality.png`

2. **Summary Tables** (TSV):
   - `qc_summary_table.tsv` - Overall metrics
   - `per_assembly_qc_summary.tsv` - Per-assembly breakdown

3. **Intermediate Data** (in `summary_stats/`):
   - All detailed statistics saved for further analysis

### Memory Usage:
- Processed 400+ assemblies in chunks
- Never loaded full dataset into memory
- All intermediate results saved to disk

### Next Steps:
1. Investigate assemblies with low concordance
2. Examine genes with coding integrity issues
3. Analyze annotation-specific genes
4. Population-level analysis using per-assembly summaries